# 3D Perception Tutorial

이번 실습에서는 (1) 3D object classification과 (2) indoor scene의 instance-level semantic segmentation을 다룹니다. 이 목표를 위해 PointCloud processing, sparse voxel convolution, 3D attention mechanism에서 WarpConvNet을 어떻게 사용하는지 기본 개념과 실제 적용 흐름을 살펴봅니다.

## 목차
1. Basic Concepts
2. PointCloud 표현
3. Voxel 표현
4. Exercise 1: Scene 선택과 Voxel Size 시각화
5. Exercise 2: SparseConv 구현
6. Exercise 3: SparseConvNet을 이용한 3D Point Cloud Semantic Segmentation
7. Exercise 4: PointTransformerV3를 이용한 3D Point Cloud Semantic Segmentation


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

# Make helper modules importable when the notebook is opened from /app or from the session folder.
for notebook_dir in [Path.cwd(), Path.cwd() / "3d-perception", Path("/app/3d-perception")]:
    if (notebook_dir / "tools.py").exists():
        os.chdir(notebook_dir)
        if str(notebook_dir) not in sys.path:
            sys.path.insert(0, str(notebook_dir))
        break
else:
    raise FileNotFoundError(
        "tools.py not found. Open the copied notebook under userN/3d-perception, "
        "or run make copy-materials before starting the notebook."
    )

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Optional, Tuple, List
import warnings
warnings.filterwarnings('ignore')

print(f"Notebook working directory: {Path.cwd()}")
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## PointCloud 표현


In [ ]:
import torch

from tools import (
    generate_sample_point_cloud,
    visualize_multiple_point_clouds,
    visualize_point_cloud,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
shapes = ['sphere', 'cube', 'torus', 'cylinder']
point_clouds = {}

for shape in shapes:
    points, features = generate_sample_point_cloud(2000, shape)
    point_clouds[shape] = points
    
# Visualize individual shape
points_sphere, features_sphere = generate_sample_point_cloud(5000, 'sphere', noise_level=0.02)
print(f"Points shape: {points_sphere.shape}")
print(f"Features shape: {features_sphere.shape}")

# Interactive visualization with color by height
fig = visualize_point_cloud(
    points_sphere, 
    colors=points_sphere[:, 2],  # Color by z-coordinate
    title="Interactive Sphere Point Cloud (5000 points)",
    point_size=2,
    colorscale='Viridis'
)

# Visualize all shapes together for comparison
visualize_multiple_point_clouds(point_clouds, title="Comparison of Different Point Cloud Shapes")


In [ ]:
from warpconvnet.geometry.types.points import Points

points_tensor = torch.from_numpy(points).to(device)
features_tensor = torch.from_numpy(features).to(device)

# Create batch indices (single batch for now)
batch_indices = torch.zeros(len(points), dtype=torch.long).to(device)

# Create PointCloud object

############## Fill here ##############
# Reference: https://github.com/NVlabs/WarpConvNet/blob/eda68fa3e3759dddadfc53d76038fd9246bbf885/warpconvnet/geometry/types/points.py
point_cloud = 
################# End #################

print(point_cloud)


## Voxel 표현


In [ ]:
from tools import visualize_voxels


## PointCloud와 Voxel 간 변환


### PointCloud -> Voxel 변환


In [ ]:
# Create sparse voxel representation with different resolutions
voxel_sizes = [0.05, 0.1, 0.2]
voxel_representations = {}

for voxel_size in voxel_sizes:
    sparse_voxels = point_cloud.to_voxels(voxel_size)
    voxel_representations[f"Size {voxel_size}"] = sparse_voxels
    visualize_voxels(sparse_voxels.coordinates, sparse_voxels.voxel_size)


### Quiz: Voxel -> PointCloud
- sparse voxel representation 하나를 다시 point cloud로 변환하세요.
- 복원된 point coordinate가 `voxel_coordinates * voxel_size`와 대응되는지 확인하세요.
- reference: https://github.com/NVlabs/WarpConvNet/blob/eda68fa3e3759dddadfc53d76038fd9246bbf885/warpconvnet/geometry/types/voxels.py


In [ ]:
# Quiz: convert sparse voxels back to point centers.
# Use the voxel representation with voxel_size = 0.1 created above.
# Hint: sparse voxels store integer coordinates. Multiplying by voxel_size gives point-space coordinates.
# Hint: check the Voxels API linked above.

voxel_to_recover = voxel_representations["Size 0.1"]

############## Fill here ##############
recovered_point_cloud = 
################# End #################

print(recovered_point_cloud)
print(f"Voxel coordinates: {voxel_to_recover.coordinate_tensor.shape}")
print(f"Recovered point coordinates: {recovered_point_cloud.coordinate_tensor.shape}")
print(f"Recovered point features: {recovered_point_cloud.feature_tensor.shape}")

visualize_point_cloud(
    recovered_point_cloud.coordinate_tensor,
    colors=recovered_point_cloud.feature_tensor[:, 0],
    title="Voxel Centers Recovered as Points",
    point_size=4,
)


## Exercise 1: Scene 선택과 Voxel Size 시각화
ScanNet scene 3개를 골라 full-scene PLY 파일로 저장합니다. 이후 voxelization 결과를 비교할 scene 이름과 voxel size 3개를 별도로 입력합니다.

### 실제 ScanNet Point Cloud 로드
위 representation 예제들은 synthetic shape를 사용했습니다. 이제 실제 ScanNet scene 하나를 로드하고, 이후 voxel-size 실험을 위해 full point cloud를 유지합니다.


In [ ]:
from pathlib import Path

import torch
from warpconvnet.geometry.types.points import Points

scannet_root = Path("/data/scannet_3d")
if not (scannet_root / "scannetv2_train.txt").exists():
    raise FileNotFoundError(
        "ScanNet data not found at /data/scannet_3d. "
        "Link the prepared ScanNet directory to /data/scannet_3d before running this notebook."
    )


def scannet_color_features_to_unit_rgb(colors):
    colors = torch.as_tensor(colors, dtype=torch.float32)
    if colors.numel() == 0:
        return colors
    if colors.min() < 0.0:
        colors = (colors + 1.0) * 0.5
    elif colors.max() > 1.0:
        colors = colors / 255.0
    return colors.clamp(0.0, 1.0)


def take_evenly_spaced_points(coords, colors, labels, max_points=None):
    if max_points is None or coords.shape[0] <= max_points:
        return coords, colors, labels
    sample_idx = torch.linspace(0, coords.shape[0] - 1, max_points).long()
    return coords[sample_idx], colors[sample_idx], labels[sample_idx]


device = globals().get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

scene_name = (scannet_root / "scannetv2_train.txt").read_text().splitlines()[0]
scene_path = scannet_root / "train" / f"{scene_name}_vh_clean_2.pth"
coords_raw, colors_raw, labels_raw = torch.load(scene_path, weights_only=False)

# Keep the full scene in CPU memory first. Downsample only after this point when
# the later demo cells need a smaller active point cloud.
coords_full = torch.as_tensor(coords_raw, dtype=torch.float32)
colors_full = scannet_color_features_to_unit_rgb(colors_raw)
labels_full = torch.as_tensor(labels_raw, dtype=torch.long)

max_scannet_points_for_demo = None  # Set an integer, e.g. 12000, to downsample after full loading.
coords_demo, colors_demo, labels_demo = take_evenly_spaced_points(
    coords_full,
    colors_full,
    labels_full,
    max_points=max_scannet_points_for_demo,
)

coords = coords_demo.to(device)
colors = colors_demo.to(device)
labels = labels_demo.to(device)

scannet_point_cloud = Points([coords], [colors])
point_cloud = scannet_point_cloud
sparse_voxels = point_cloud.to_voxels(0.1)

print(f"Loaded ScanNet scene: {scene_name}")
print(f"Full scene points: {coords_full.shape[0]:,}")
print(f"Active demo points: {point_cloud.coordinate_tensor.shape[0]:,}")
print(f"Sparse voxels at voxel_size=0.1: {sparse_voxels.coordinate_tensor.shape[0]:,}")
print(f"Semantic labels: {labels.shape}")


In [ ]:
from pathlib import Path

from tools import save_scannet_scene_as_ply

# Choose three ScanNet scenes and save each full scene with RGB and semantic colors.
scene_names_to_export = [
    ...,  # choose a ScanNet scene name, e.g. "scene0439_01"
    ...,
    ...,
]
split_to_export = None  # None searches train/val/test automatically. Or set "train", "val", or "test".
max_export_points = None  # None saves each full scene.
ply_output_dir = Path("ply_exports")
full_scene_color_modes = ["rgb", "semantic"]

full_scene_ply_paths = []
for scene_name_to_export in scene_names_to_export:
    for color_mode in full_scene_color_modes:
        ply_path = save_scannet_scene_as_ply(
            scene_name_to_export,
            split=split_to_export,
            max_points=max_export_points,
            output_dir=ply_output_dir,
            color_mode=color_mode,
            voxel_size=None,
        )
        full_scene_ply_paths.append(ply_path)

full_scene_ply_paths


### Voxelize할 Scene과 Voxel Size 선택
Voxelization 결과를 비교할 ScanNet scene 이름을 직접 입력하고 voxel size 3개를 고릅니다. 같은 scene을 서로 다른 voxel resolution으로 세 번 저장합니다.


In [ ]:
# Choose a ScanNet scene and three voxel sizes for visual comparison.
voxelized_scene_name_to_export = ...  # e.g. "scene0439_01"
voxelized_split_to_export = None  # None searches train/val/test automatically. Or set "train", "val", or "test".
voxelized_max_export_points = max_export_points
voxelized_ply_output_dir = ply_output_dir
voxelized_export_sizes = [
    ...,  # choose the first voxel size, e.g. 0.05
    ...,  # choose the second voxel size
    ...,  # choose the third voxel size
]
voxelized_export_color_modes = ["rgb", "semantic"]


In [ ]:
from tools import load_scannet_scene_arrays, sample_scene_arrays, voxelize_scene_arrays

# Count how many occupied voxels remain for the selected scene at each voxel size.
loaded_split, selected_coords, selected_colors, selected_labels = load_scannet_scene_arrays(
    voxelized_scene_name_to_export,
    split=voxelized_split_to_export,
    root=scannet_root,
)
selected_coords, selected_colors, selected_labels = sample_scene_arrays(
    selected_coords,
    selected_colors,
    selected_labels,
    max_points=voxelized_max_export_points,
)

voxel_size_summary = []
for voxel_size in voxelized_export_sizes:
    voxel_coords, _, _ = voxelize_scene_arrays(
        selected_coords,
        selected_colors,
        selected_labels,
        voxel_size=voxel_size,
    )
    occupied_voxels = voxel_coords.shape[0]
    voxel_size_summary.append({
        "voxel_size": voxel_size,
        "occupied_voxels": occupied_voxels,
        "compression_ratio": selected_coords.shape[0] / occupied_voxels,
    })

print(f"Selected scene: {voxelized_scene_name_to_export} ({loaded_split})")
print(f"Full scene points: {selected_coords.shape[0]:,}")
print("voxel_size | occupied_voxels | points_per_voxel")
for row in voxel_size_summary:
    print(
        f"{row['voxel_size']:>10.2f} | "
        f"{row['occupied_voxels']:>15,} | "
        f"{row['compression_ratio']:>16.2f}"
    )


In [ ]:
# Save the selected scene at the selected voxelization sizes.
voxelized_ply_paths = []
for voxel_size in voxelized_export_sizes:
    for color_mode in voxelized_export_color_modes:
        ply_path = save_scannet_scene_as_ply(
            voxelized_scene_name_to_export,
            split=voxelized_split_to_export,
            max_points=voxelized_max_export_points,
            output_dir=voxelized_ply_output_dir,
            color_mode=color_mode,
            voxel_size=voxel_size,
        )
        voxelized_ply_paths.append(ply_path)

voxelized_ply_paths


In [ ]:
# Exercise: plot용 voxel size 구간을 직접 정하고 point 수와 occupied voxel 수를 그래프로 그려보세요.
# PLY 저장에 사용한 voxelized_export_sizes와 별개로 더 촘촘한 구간을 사용할 수 있습니다.
import matplotlib.pyplot as plt

plot_voxel_sizes = [
    ...,  # e.g. 0.02
    ...,
    ...,
]
plot_voxel_size_summary = []
for voxel_size in plot_voxel_sizes:
    voxel_coords, _, _ = voxelize_scene_arrays(
        selected_coords,
        selected_colors,
        selected_labels,
        voxel_size=voxel_size,
    )
    plot_voxel_size_summary.append({
        "voxel_size": voxel_size,
        "raw_points": selected_coords.shape[0],
        "occupied_voxels": voxel_coords.shape[0],
    })

# plot_voxel_size_summary에서 그래프에 사용할 x/y 값을 꺼내세요.
voxel_sizes_for_plot = ...
raw_point_counts_for_plot = ...
occupied_voxel_counts_for_plot = ...

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(
    voxel_sizes_for_plot,
    raw_point_counts_for_plot,
    marker="o",
    label="Raw points",
)
ax.plot(
    voxel_sizes_for_plot,
    occupied_voxel_counts_for_plot,
    marker="s",
    label="Occupied voxels",
)
ax.set_xlabel("Voxel size")
ax.set_ylabel("Count")
ax.set_title(f"Point / occupied voxel count: {voxelized_scene_name_to_export}")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


### ScanNet Semantic Color Legend
semantic-label-colored PLY 파일을 볼 때 아래 legend를 참고하세요.


In [ ]:
from tools import display_scannet_semantic_legend

display_scannet_semantic_legend()


## Exercise 2: SparseConv 구현
WarpConvNet을 reference로 사용하고, submanifold sparse convolution 하나를 plain PyTorch-style tensor operation으로 직접 구현한 뒤 output feature를 수치적으로 비교합니다.

### Submanifold Sparse Convolution이란?
Dense 3D convolution은 dense grid의 모든 voxel을 방문합니다. empty space도 포함됩니다. 반면 sparse convolution은 occupied voxel만 저장하고 계산합니다. Point cloud에서는 3D grid 대부분이 비어 있기 때문에 이 차이가 중요합니다.

<table>
  <tr>
    <td align="center"><img src="assets/dense_tensor.gif" width="360"></td>
    <td align="center"><img src="assets/sparse_tensor.gif" width="360"></td>
  </tr>
  <tr>
    <td align="center"><b>Dense tensor</b><br>모든 grid cell을 저장하고 방문합니다.</td>
    <td align="center"><b>Sparse tensor</b><br>occupied cell만 저장하고 계산합니다.</td>
  </tr>
</table>

이번 exercise에서는 **submanifold** sparse convolution을 구현합니다. submanifold라는 말은 output이 input과 정확히 같은 active voxel coordinate를 유지한다는 뜻입니다. Convolution은 각 active voxel feature를 주변 active voxel을 보며 업데이트하지만, 비어 있는 위치에 새로운 active voxel을 만들지는 않습니다.

각 active voxel은 다음을 가집니다:

- integer coordinate `(batch_index, x, y, z)`
- input feature vector `features[input_index]`
- output feature vector `out_features[output_index]`

`3 x 3 x 3` kernel에서는 각 output voxel이 가능한 neighbor offset 27개를 확인합니다:

```text
dx in [-1, 0, 1]
dy in [-1, 0, 1]
dz in [-1, 0, 1]
```

각 output voxel coordinate `(b, x, y, z)`에 대해 구현 흐름은 다음과 같습니다:

```text
1. Lookup table 만들기:

   coord_to_index[(b, x, y, z)] = 해당 active voxel의 index

2. 모든 output voxel에 대해:

   output_coord = (b, x, y, z)
   output_feature = zeros(out_channels)

3. 모든 kernel offset에 대해:

   neighbor_coord = (b, x + dx, y + dy, z + dz)
   input_index = coord_to_index.get(neighbor_coord)

   input_index가 없으면:
       해당 neighbor voxel은 empty이므로 건너뜁니다

   input_index가 있으면:
       input_feature = features[input_index]
       kernel_weight = weight[kernel_index]
       contribution = input_feature @ kernel_weight
       output_feature = output_feature + contribution

4. 모든 neighbor contribution을 누적한 뒤:

   bias가 있으면:
       output_feature = output_feature + bias
```

핵심 식은 다음과 같습니다:

```text
out_feature_at_voxel = sum(active_neighbor_feature @ matching_kernel_weight) + bias
```

다음 셀은 WarpConvNet의 optimized `SparseConv3d`를 실행합니다. 그 다음 셀에서는 같은 operation을 explicit Python loop로 구현해서 coordinate lookup과 neighbor accumulation이 보이도록 합니다.


In [ ]:
# 1. WarpConvNet SparseConv3d baseline.
# Use a small deterministic sparse voxel tensor so the custom implementation can be checked exactly.
import torch

from warpconvnet.geometry.types.voxels import Voxels
from warpconvnet.nn.modules.sparse_conv import SparseConv3d


device = globals().get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

reference_coords = torch.tensor([
    [0, 0, 0],
    [1, 0, 0],
    [-1, 0, 0],
    [0, 1, 0],
    [0, -1, 0],
    [0, 0, 1],
    [0, 0, -1],
    [1, 1, 0],
    [1, 0, 1],
], dtype=torch.int32, device=device)

num_reference_voxels = reference_coords.shape[0]
feature_values = torch.arange(num_reference_voxels * 3, dtype=torch.float32, device=device)
feature_values = feature_values.reshape(num_reference_voxels, 3)
reference_features = (feature_values % 5) / 100.0
reference_voxels = Voxels([reference_coords], [reference_features])

warp_sparse_conv = SparseConv3d(
    3,
    4,
    kernel_size=3,
    stride=1,
    bias=True,
    fwd_algo="explicit_gemm",
    compute_dtype=torch.float32,
).to(device)

with torch.no_grad():
    weight_values = torch.linspace(
        -0.02,
        0.02,
        warp_sparse_conv.weight.numel(),
        device=device,
    )
    weight_values = weight_values.reshape_as(warp_sparse_conv.weight)
    warp_sparse_conv.weight.copy_(weight_values)

    bias_values = torch.linspace(
        -0.01,
        0.01,
        warp_sparse_conv.bias.numel(),
        device=device,
    )
    warp_sparse_conv.bias.copy_(bias_values)

warp_output = warp_sparse_conv(reference_voxels)

print(f"Input coordinates: {reference_voxels.coordinate_tensor.shape}")
print(f"Input features: {reference_voxels.feature_tensor.shape}")
print(f"WarpConvNet output features: {warp_output.feature_tensor.shape}")
print(warp_output.feature_tensor[:3].detach().cpu())


In [ ]:
# 2. Custom reference implementation of submanifold 3D sparse convolution.
# Fill this in, then run the next cell to compare it with WarpConvNet.
def custom_submanifold_sparse_conv3d(input_voxels, weight, bias=None):
    if weight.shape[0] != 27:
        raise ValueError("This reference implementation expects a 3x3x3 kernel.")

    coords = input_voxels.batch_indexed_coordinates.to(torch.int64)
    coords_list = coords.detach().cpu().tolist()
    features = input_voxels.feature_tensor

    num_voxels = features.shape[0]
    out_channels = weight.shape[2]
    out_features = torch.zeros(
        (num_voxels, out_channels),
        dtype=features.dtype,
        device=features.device,
    )

    coord_to_index = {}
    for point_index in range(num_voxels):
        batch_index = coords_list[point_index][0]
        x = coords_list[point_index][1]
        y = coords_list[point_index][2]
        z = coords_list[point_index][3]
        coord_to_index[(batch_index, x, y, z)] = point_index

    kernel_offsets = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:
                kernel_offsets.append((dx, dy, dz))

    ############## Fill here ##############
    for output_index in range(num_voxels):
        batch_index = coords_list[output_index][0]
        x = coords_list[output_index][1]
        y = coords_list[output_index][2]
        z = coords_list[output_index][3]

        for kernel_index in range(len(kernel_offsets)):
            dx, dy, dz = kernel_offsets[kernel_index]
            neighbor_coord = ...
            input_index = ...

            if input_index is None:
                continue

            input_feature = ...
            kernel_weight = ...
            contribution = ...
            out_features[output_index] = ...

    if bias is not None:
        out_features = ...
    ################# End #################

    return input_voxels.replace(batched_features=out_features)


In [ ]:
# 3. Compare the custom implementation with WarpConvNet.
custom_output = custom_submanifold_sparse_conv3d(
    reference_voxels,
    warp_sparse_conv.weight,
    warp_sparse_conv.bias,
)

coord_match = torch.equal(
    warp_output.batch_indexed_coordinates.detach().cpu(),
    custom_output.batch_indexed_coordinates.detach().cpu(),
)
max_abs_error = (warp_output.feature_tensor - custom_output.feature_tensor).abs().max()
mean_abs_error = (warp_output.feature_tensor - custom_output.feature_tensor).abs().mean()

print(f"Coordinate match: {coord_match}")
print(f"Max feature error: {max_abs_error.item():.8f}")
print(f"Mean feature error: {mean_abs_error.item():.8f}")
print("WarpConvNet output sample:")
print(warp_output.feature_tensor[:3].detach().cpu())
print("Custom output sample:")
print(custom_output.feature_tensor[:3].detach().cpu())

assert coord_match
assert torch.allclose(warp_output.feature_tensor, custom_output.feature_tensor, atol=1e-6, rtol=1e-6)


## Exercise 3: SparseConvNet을 이용한 3D Point Cloud Semantic Segmentation


In [ ]:
from typing import Dict, List, Optional, Tuple, Union
import os
from pathlib import Path
import yaml

try:
    import hydra
    from hydra.core.config_store import ConfigStore
    from omegaconf import DictConfig, OmegaConf
except ImportError:
    print("Hydra and OmegaConf not installed, pip install hydra-core omegaconf")
    exit(1)

import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import warp as wp
from torch import Tensor
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Subset
from torchmetrics.classification import MulticlassConfusionMatrix
from tqdm import tqdm
from warpconvnet.dataset.scannet import ScanNetDataset
from warpconvnet.geometry.base.geometry import Geometry
from warpconvnet.geometry.types.points import Points
from warpconvnet.nn.modules.sparse_pool import PointToSparseWrapper

from mink_unet import MinkUNetBase


In [ ]:
def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def validate_scannet_data_dir(data_dir: Union[str, Path]) -> str:
    path = Path(data_dir).expanduser()
    if not path.is_absolute():
        path = (Path.cwd() / path).resolve()

    required_files = ["scannetv2_train.txt", "scannetv2_val.txt"]
    missing_files = [name for name in required_files if not (path / name).exists()]
    if missing_files:
        raise FileNotFoundError(
            f"ScanNet data not found at {path}. "
            "Link the prepared ScanNet directory to /data/scannet_3d before running this notebook. "
            f"Missing files: {missing_files}"
        )
    return str(path)

def collate_fn(batch: List[Dict[str, Tensor]]):
    """
    Return dict of list of tensors
    """
    keys = batch[0].keys()
    return {key: [torch.tensor(item[key]) for item in batch] for key in keys}


class DataToTensor:
    def __init__(
        self,
        device: str = "cuda",
    ):
        self.device = device

    def __call__(self, batch_dict: Dict[str, Tensor]) -> Tuple[Geometry, Dict[str, Tensor]]:
        # cat all features into a single tensor
        cat_batch_dict = {k: torch.cat(v, dim=0).to(self.device) for k, v in batch_dict.items()}
        return (
            Points.from_list_of_coordinates(
                batch_dict["coords"],
                features=batch_dict["colors"],
            ).to(self.device),
            cat_batch_dict,
        )


def confusion_matrix_to_metrics(conf_matrix: Tensor) -> Dict[str, float]:
    """
    Return accuracy, miou, class_iou, class_accuracy

    Rows are ground truth, columns are predictions.
    """
    conf_matrix = conf_matrix.cpu()
    true_positive = conf_matrix.diag()
    total_points = conf_matrix.sum()
    gt_count = conf_matrix.sum(dim=1)
    pred_count = conf_matrix.sum(dim=0)
    union = gt_count + pred_count - true_positive

    accuracy = (true_positive.sum() / total_points).item() * 100 if total_points > 0 else 0.0

    class_accuracy = true_positive / gt_count.clamp_min(1) * 100
    class_accuracy[gt_count == 0] = torch.nan

    valid_iou = union > 0
    raw_class_iou = true_positive / union.clamp_min(1)
    miou = raw_class_iou[valid_iou].mean().item() * 100 if valid_iou.any().item() else 0.0

    class_iou = raw_class_iou.clone()
    class_iou[~valid_iou] = torch.nan
    return {
        "accuracy": accuracy,
        "miou": miou,
        "class_iou": class_iou,
        "class_accuracy": class_accuracy,
    }

from tools import save_latest_visual_data_as_plys

def checkpoint_path_from_cfg(cfg, filename: str) -> Path:
    output_dir = Path(cfg.paths.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir / filename


def save_model_checkpoint(model: nn.Module, cfg, filename: str) -> Path:
    checkpoint_path = checkpoint_path_from_cfg(cfg, filename)
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "config": OmegaConf.to_container(cfg, resolve=True),
    }
    torch.save(checkpoint, checkpoint_path)
    print(f"Saved checkpoint: {checkpoint_path}")
    return checkpoint_path


def load_model_checkpoint(model: nn.Module, checkpoint_path: Union[str, Path], device: torch.device):
    checkpoint_path = Path(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state_dict)
    print(f"Loaded checkpoint: {checkpoint_path}")
    return checkpoint



In [ ]:
def _cfg_get(node, key: str, default):
    return node[key] if key in node else default


def _use_amp(cfg: DictConfig, section: str) -> bool:
    node = cfg[section]
    fallback = _cfg_get(cfg.train, "amp", True)
    enabled = _cfg_get(node, "amp", fallback)
    return bool(enabled) and str(cfg.device).startswith("cuda")


def _check_output_and_labels(output: Geometry, labels: Tensor):
    if output.features.shape[0] != labels.shape[0]:
        raise RuntimeError(
            f"Model output and labels have different lengths: "
            f"{output.features.shape[0]} vs {labels.shape[0]}"
        )


def train(
    model: nn.Module,
    train_loader: DataLoader,
    optimizer: optim.Optimizer,
    epoch: int,
    cfg: DictConfig,
):
    model.train()
    bar = tqdm(train_loader)
    dict_to_data = DataToTensor(device=cfg.device)
    amp_enabled = _use_amp(cfg, "train")
    max_grad_norm = float(_cfg_get(cfg.train, "max_grad_norm", 0.0))

    for batch_dict in bar:
        optimizer.zero_grad(set_to_none=True)
        st, batch_dict = dict_to_data(batch_dict)
        labels = batch_dict["labels"].long().to(cfg.device)

        with torch.amp.autocast(device_type="cuda", enabled=amp_enabled):
            output = model(st.to(cfg.device))
            _check_output_and_labels(output, labels)
            loss = F.cross_entropy(
                output.features,
                labels,
                reduction="mean",
                ignore_index=cfg.data.ignore_index,
            )

        if not torch.isfinite(loss).item():
            raise FloatingPointError(f"Non-finite training loss: {loss.item()}")

        loss.backward()
        if max_grad_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        bar.set_description(f"Train Epoch: {epoch} Loss: {loss.item(): .3f}")


@torch.inference_mode()
def test(
    model: nn.Module,
    test_loader: DataLoader,
    cfg: DictConfig,
    num_test_batches: Optional[int] = None,
    save_visuals: bool = False,
    return_visuals: bool = False,
):
    model.eval()
    torch.cuda.empty_cache()
    confusion_matrix = MulticlassConfusionMatrix(
        num_classes=cfg.data.num_classes, ignore_index=cfg.data.ignore_index
    ).to(cfg.device)
    test_loss = 0.0
    num_batches = 0

    visual_data = []
    dict_to_data = DataToTensor(device=cfg.device)
    amp_enabled = _use_amp(cfg, "test")

    for batch_dict in tqdm(test_loader):
        original_batch_dict = batch_dict
        st, batch_dict = dict_to_data(batch_dict)
        labels = batch_dict["labels"].long().to(cfg.device)

        with torch.amp.autocast(device_type="cuda", enabled=amp_enabled):
            output = model(st.to(cfg.device))
            _check_output_and_labels(output, labels)
            loss = F.cross_entropy(
                output.features,
                labels,
                reduction="mean",
                ignore_index=cfg.data.ignore_index,
            )

        if torch.isfinite(loss).item():
            test_loss += loss.item()
        pred = output.features.argmax(dim=1)
        confusion_matrix.update(pred, labels)

        if save_visuals or return_visuals:
            num_items_in_batch = len(st.offsets) - 1
            for i in range(num_items_in_batch):
                start_idx = st.offsets[i]
                end_idx = st.offsets[i + 1]

                visual_data.append({
                    "coords": original_batch_dict["coords"][i],
                    "colors": original_batch_dict["colors"][i],
                    "preds": pred[start_idx:end_idx].cpu(),
                    "labels": labels[start_idx:end_idx].cpu(),
                })

        num_batches += 1
        if num_test_batches is not None and num_batches >= num_test_batches:
            break

    metrics = confusion_matrix_to_metrics(confusion_matrix.compute())
    average_loss = test_loss / max(num_batches, 1)
    print(
        f"Test set: Average loss: {average_loss: .4f}, "
        f"Accuracy: {metrics['accuracy']: .2f}%, mIoU: {metrics['miou']: .2f}%"
    )
    if return_visuals:
        return metrics, visual_data
    return metrics


In [ ]:
# Embedded YAML configuration
CONFIG_YAML_BASE = """
# Path configuration
paths:
  data_dir: /data/scannet_3d
  output_dir: ./results/scannet_3d/base_sparseconv
  ckpt_path: null

# Training configuration.
train:
  batch_size: 4
  lr: 0.001
  epochs: 2
  checkpoint_interval: 5
  step_size: 20
  gamma: 0.7
  num_workers: 8

# Testing configuration
test:
  batch_size: 5
  num_workers: 4

# Dataset configuration
data:
  num_classes: 20
  voxel_size: 0.02
  ignore_index: 255

# Model configuration
model:
  _target_: mink_unet.MinkUNet18
  in_channels: 3
  out_channels: 20
  in_type: "voxel"

# General configuration
device: "cuda"
use_wandb: false
seed: 42
"""


In [ ]:
# Load configs
cell_start_time = time.time()
cfg_dict = yaml.safe_load(CONFIG_YAML_BASE)
cfg = OmegaConf.create(cfg_dict)
cfg.paths.data_dir = validate_scannet_data_dir(cfg.paths.data_dir)
print(f"Using ScanNet data: {cfg.paths.data_dir}")

set_seed(cfg.seed)
device = torch.device(cfg.device)

# Define dataloaders
train_dataset = ScanNetDataset(cfg.paths.data_dir, split="train",)
train_dataset = train_dataset
test_dataset = ScanNetDataset(cfg.paths.data_dir, split="val",)
test_dataset = test_dataset
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.train.batch_size,
    num_workers=cfg.train.num_workers,
    shuffle=True,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.test.batch_size,
    num_workers=cfg.test.num_workers,
    shuffle=False,
    collate_fn=collate_fn,
)

# Model initialization
model = MinkUNetBase(
    in_channels=cfg.model.in_channels,
    out_channels=cfg.model.out_channels,
).to(device)

if hasattr(cfg.model, "in_type") and cfg.model.in_type == "voxel":
    model = PointToSparseWrapper(
        inner_module=model,
        voxel_size=cfg.data.voxel_size,
        concat_unpooled_pc=False,
    )

optimizer = optim.AdamW(model.parameters(), lr=cfg.train.lr)
scheduler = StepLR(optimizer, step_size=cfg.train.step_size, gamma=cfg.train.gamma)
checkpoint_interval = int(getattr(cfg.train, "checkpoint_interval", 5))

# Test before training
# metrics = test(model, test_loader, cfg, num_test_batches=5)

for epoch in range(1, cfg.train.epochs + 1):
    train(
        model,
        train_loader,
        optimizer,
        epoch,
        cfg,
    )
    metrics = test(model, test_loader, cfg)
    if checkpoint_interval > 0 and epoch % checkpoint_interval == 0 and epoch < cfg.train.epochs:
        save_model_checkpoint(model, cfg, f"base_sparseconv_epoch_{epoch:03d}.pth")
    scheduler.step()

print(f"Final mIoU: {metrics['miou']: .2f}%")

checkpoint_path = save_model_checkpoint(model, cfg, "base_sparseconv_final.pth")
print(f"Training cell runtime: {(time.time() - cell_start_time) / 60:.1f} min")

del model
torch.cuda.empty_cache()


In [ ]:
# Load the saved Base SparseConvNet checkpoint and run inference.
cfg_dict = yaml.safe_load(CONFIG_YAML_BASE)
cfg = OmegaConf.create(cfg_dict)
cfg.paths.data_dir = validate_scannet_data_dir(cfg.paths.data_dir)
print(f"Using ScanNet data: {cfg.paths.data_dir}")

set_seed(cfg.seed)
device = torch.device(cfg.device)

test_dataset = ScanNetDataset(cfg.paths.data_dir, split="val",)
test_dataset = Subset(test_dataset, range(8))
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.test.batch_size,
    num_workers=cfg.test.num_workers,
    shuffle=False,
    collate_fn=collate_fn,
)

model = MinkUNetBase(
    in_channels=cfg.model.in_channels,
    out_channels=cfg.model.out_channels,
).to(device)

if hasattr(cfg.model, "in_type") and cfg.model.in_type == "voxel":
    model = PointToSparseWrapper(
        inner_module=model,
        voxel_size=cfg.data.voxel_size,
        concat_unpooled_pc=False,
    )

checkpoint_path = checkpoint_path_from_cfg(cfg, "base_sparseconv_final.pth")
load_model_checkpoint(model, checkpoint_path, device)
metrics, latest_visual_data = test(model, test_loader, cfg, return_visuals=True)
print(f"Loaded-checkpoint mIoU: {metrics['miou']: .2f}%")

print(f"Collected visual data for {len(latest_visual_data)} point clouds")

del model
torch.cuda.empty_cache()


In [ ]:
# Run this after a checkpoint inference cell has collected latest_visual_data.
from pathlib import Path

# Change this directory to custom_sparseconv or point_transformer_v3 for other model outputs.
visual_output_dir = Path("./results/scannet_3d/base_sparseconv")
max_visualized_samples = 4

semantic_visual_ply_paths = save_latest_visual_data_as_plys(
    latest_visual_data,
    output_dir=visual_output_dir,
    max_items=max_visualized_samples,
)
semantic_visual_ply_paths


In [ ]:
class MinkUNetCustom(MinkUNetBase):
    def __init__(self, in_channels: int, out_channels: int, **kwargs):
        raise NotImplemented


In [ ]:
# Embedded YAML configuration
CONFIG_YAML_CUSTOM = """
# Path configuration
paths:
  data_dir: /data/scannet_3d
  output_dir: ./results/scannet_3d/custom_sparseconv
  ckpt_path: null

# Training configuration.
train:
  batch_size: 64
  lr: 0.005
  epochs: 2
  checkpoint_interval: 5
  step_size: 20
  gamma: 0.7
  num_workers: 8

# Testing configuration
test:
  batch_size: 12
  num_workers: 4

# Dataset configuration
data:
  num_classes: 20
  voxel_size: 0.02
  ignore_index: 255

# Model configuration
model:
  _target_: mink_unet.MinkUNet18
  in_channels: 3
  out_channels: 20
  in_type: "voxel"

# General configuration
device: "cuda"
use_wandb: false
seed: 42
"""


In [ ]:
# Load configs
cell_start_time = time.time()
cfg_dict = yaml.safe_load(CONFIG_YAML_CUSTOM)
cfg = OmegaConf.create(cfg_dict)
cfg.paths.data_dir = validate_scannet_data_dir(cfg.paths.data_dir)
print(f"Using ScanNet data: {cfg.paths.data_dir}")

set_seed(cfg.seed)
device = torch.device(cfg.device)

# Define dataloaders
train_dataset = ScanNetDataset(cfg.paths.data_dir, split="train",)
test_dataset = ScanNetDataset(cfg.paths.data_dir, split="val",)
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.train.batch_size,
    num_workers=cfg.train.num_workers,
    shuffle=True,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.test.batch_size,
    num_workers=cfg.test.num_workers,
    shuffle=False,
    collate_fn=collate_fn,
)

# Model initialization
model = MinkUNetCustom(
    in_channels=cfg.model.in_channels,
    out_channels=cfg.model.out_channels,
).to(device)

if hasattr(cfg.model, "in_type") and cfg.model.in_type == "voxel":
    model = PointToSparseWrapper(
        inner_module=model,
        voxel_size=cfg.data.voxel_size,
        concat_unpooled_pc=False,
    )

optimizer = optim.AdamW(model.parameters(), lr=cfg.train.lr)
scheduler = StepLR(optimizer, step_size=cfg.train.step_size, gamma=cfg.train.gamma)
checkpoint_interval = int(getattr(cfg.train, "checkpoint_interval", 5))

# Test before training
metrics = test(model, test_loader, cfg, num_test_batches=5)

for epoch in range(1, cfg.train.epochs + 1):
    train(
        model,
        train_loader,
        optimizer,
        epoch,
        cfg,
    )
    metrics = test(model, test_loader, cfg)
    if checkpoint_interval > 0 and epoch % checkpoint_interval == 0 and epoch < cfg.train.epochs:
        save_model_checkpoint(model, cfg, f"custom_sparseconv_epoch_{epoch:03d}.pth")
    scheduler.step()

print(f"Final mIoU: {metrics['miou']: .2f}%")

checkpoint_path = save_model_checkpoint(model, cfg, "custom_sparseconv_final.pth")
print(f"Training cell runtime: {(time.time() - cell_start_time) / 60:.1f} min")

del model
torch.cuda.empty_cache()


In [ ]:
# Load the saved Custom SparseConvNet checkpoint and run inference.
cfg_dict = yaml.safe_load(CONFIG_YAML_CUSTOM)
cfg = OmegaConf.create(cfg_dict)
cfg.paths.data_dir = validate_scannet_data_dir(cfg.paths.data_dir)
print(f"Using ScanNet data: {cfg.paths.data_dir}")

set_seed(cfg.seed)
device = torch.device(cfg.device)

test_dataset = ScanNetDataset(cfg.paths.data_dir, split="val",)
test_dataset = Subset(test_dataset, range(8))
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.test.batch_size,
    num_workers=cfg.test.num_workers,
    shuffle=False,
    collate_fn=collate_fn,
)

model = MinkUNetCustom(
    in_channels=cfg.model.in_channels,
    out_channels=cfg.model.out_channels,
).to(device)

if hasattr(cfg.model, "in_type") and cfg.model.in_type == "voxel":
    model = PointToSparseWrapper(
        inner_module=model,
        voxel_size=cfg.data.voxel_size,
        concat_unpooled_pc=False,
    )

checkpoint_path = checkpoint_path_from_cfg(cfg, "custom_sparseconv_final.pth")
load_model_checkpoint(model, checkpoint_path, device)
metrics, latest_visual_data = test(model, test_loader, cfg, return_visuals=True)
print(f"Loaded-checkpoint mIoU: {metrics['miou']: .2f}%")

print(f"Collected visual data for {len(latest_visual_data)} point clouds")

del model
torch.cuda.empty_cache()


## Exercise 4: PointTransformerV3를 이용한 3D Point Cloud Semantic Segmentation


In [ ]:
# Embedded YAML configuration
CONFIG_YAML_PTV3 = """
# Path configuration
paths:
  data_dir: /data/scannet_3d
  output_dir: ./results/scannet_3d/point_transformer_v3
  ckpt_path: null

# Training configuration.
train:
  batch_size: 1
  lr: 0.0001
  epochs: 2
  checkpoint_interval: 5
  step_size: 20
  gamma: 0.7
  num_workers: 4
  amp: false
  max_grad_norm: 1.0

# Testing configuration
test:
  batch_size: 1
  num_workers: 2
  amp: false

# Dataset configuration
data:
  num_classes: 20
  voxel_size: 0.02
  ignore_index: 255

# Model configuration
model:
  _target_: point_transformer_v3.PointTransformerV3
  in_channels: 3
  out_channels: 20
  in_type: "voxel"
  shuffle_orders: false
  drop_path: 0.0

# General configuration
device: "cuda"
use_wandb: false
seed: 42
"""


In [ ]:
from point_transformer_v3 import PointTransformerV3

# Load configs
cell_start_time = time.time()
cfg_dict = yaml.safe_load(CONFIG_YAML_PTV3)
cfg = OmegaConf.create(cfg_dict)
cfg.paths.data_dir = validate_scannet_data_dir(cfg.paths.data_dir)
print(f"Using ScanNet data: {cfg.paths.data_dir}")

set_seed(cfg.seed)
device = torch.device(cfg.device)

# Define dataloaders
train_dataset = ScanNetDataset(cfg.paths.data_dir, split="train",)
test_dataset = ScanNetDataset(cfg.paths.data_dir, split="val",)
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.train.batch_size,
    num_workers=cfg.train.num_workers,
    shuffle=True,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.test.batch_size,
    num_workers=cfg.test.num_workers,
    shuffle=False,
    collate_fn=collate_fn,
)

# Model initialization
ptv3_kwargs = {}
if "shuffle_orders" in cfg.model:
    ptv3_kwargs["shuffle_orders"] = bool(cfg.model.shuffle_orders)
if "drop_path" in cfg.model:
    ptv3_kwargs["drop_path"] = float(cfg.model.drop_path)

model = PointTransformerV3(
    in_channels=cfg.model.in_channels,
    out_channels=cfg.model.out_channels,
    **ptv3_kwargs,
).to(device)

if hasattr(cfg.model, "in_type") and cfg.model.in_type == "voxel":
    model = PointToSparseWrapper(
        inner_module=model,
        voxel_size=cfg.data.voxel_size,
        concat_unpooled_pc=False,
    )

optimizer = optim.AdamW(model.parameters(), lr=cfg.train.lr)
scheduler = StepLR(optimizer, step_size=cfg.train.step_size, gamma=cfg.train.gamma)
checkpoint_interval = int(getattr(cfg.train, "checkpoint_interval", 5))

# Test before training
metrics = test(model, test_loader, cfg, num_test_batches=5)

for epoch in range(1, cfg.train.epochs + 1):
    train(
        model,
        train_loader,
        optimizer,
        epoch,
        cfg,
    )
    metrics = test(model, test_loader, cfg)
    if checkpoint_interval > 0 and epoch % checkpoint_interval == 0 and epoch < cfg.train.epochs:
        save_model_checkpoint(model, cfg, f"ptv3_epoch_{epoch:03d}.pth")
    scheduler.step()

print(f"Final mIoU: {metrics['miou']: .2f}%")

checkpoint_path = save_model_checkpoint(model, cfg, "ptv3_final.pth")
print(f"Training cell runtime: {(time.time() - cell_start_time) / 60:.1f} min")

del model
torch.cuda.empty_cache()


In [ ]:
# Load the saved PointTransformerV3 checkpoint and run inference.
from point_transformer_v3 import PointTransformerV3

cfg_dict = yaml.safe_load(CONFIG_YAML_PTV3)
cfg = OmegaConf.create(cfg_dict)
cfg.paths.data_dir = validate_scannet_data_dir(cfg.paths.data_dir)
print(f"Using ScanNet data: {cfg.paths.data_dir}")

set_seed(cfg.seed)
device = torch.device(cfg.device)

test_dataset = ScanNetDataset(cfg.paths.data_dir, split="val",)
test_dataset = Subset(test_dataset, range(8))
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.test.batch_size,
    num_workers=cfg.test.num_workers,
    shuffle=False,
    collate_fn=collate_fn,
)

ptv3_kwargs = {}
if "shuffle_orders" in cfg.model:
    ptv3_kwargs["shuffle_orders"] = bool(cfg.model.shuffle_orders)
if "drop_path" in cfg.model:
    ptv3_kwargs["drop_path"] = float(cfg.model.drop_path)

model = PointTransformerV3(
    in_channels=cfg.model.in_channels,
    out_channels=cfg.model.out_channels,
    **ptv3_kwargs,
).to(device)

if hasattr(cfg.model, "in_type") and cfg.model.in_type == "voxel":
    model = PointToSparseWrapper(
        inner_module=model,
        voxel_size=cfg.data.voxel_size,
        concat_unpooled_pc=False,
    )

checkpoint_path = checkpoint_path_from_cfg(cfg, "ptv3_final.pth")
load_model_checkpoint(model, checkpoint_path, device)
metrics, latest_visual_data = test(model, test_loader, cfg, return_visuals=True)
print(f"Loaded-checkpoint mIoU: {metrics['miou']: .2f}%")

print(f"Collected visual data for {len(latest_visual_data)} point clouds")

del model
torch.cuda.empty_cache()
